# 프로젝트 2 - Weekend 1: 정답지

각 문제의 풀이 코드입니다.

---
## 문제 1: ETF 사용자 페르소나 정의

In [ ]:
# ✅ 문제 1 정답
personas = {
    "초보": {
        "description": "투자 경험이 거의 없고, 원금 보존을 최우선시하는 투자자",
        "queries": [
            {"query": "안전한 투자 상품 추천해주세요", "expected_category": ["채권", "머니마켓"]},
            {"query": "원금 손실 위험 없는 ETF", "expected_category": ["머니마켓"]},
            {"query": "적금보다 나은 안전한 투자", "expected_category": ["채권", "배당"]},
        ]
    },
    "중급": {
        "description": "기본적인 투자 지식이 있고, 적절한 위험을 감수하며 자산 증식을 추구하는 투자자",
        "queries": [
            {"query": "S&P500 추종 ETF 비교", "expected_category": ["해외주식"]},
            {"query": "배당과 성장 균형 잡힌 포트폴리오", "expected_category": ["배당", "국내주식"]},
            {"query": "분산투자 가능한 ETF 조합", "expected_category": ["국내주식", "해외주식", "채권"]},
        ]
    },
    "전문": {
        "description": "투자 경험이 풍부하고, 높은 수익을 위해 높은 변동성을 감수할 수 있는 투자자",
        "queries": [
            {"query": "AI 반도체 관련 ETF 섹터 분석", "expected_category": ["테마", "해외주식"]},
            {"query": "레버리지 ETF 단기 트레이딩 전략", "expected_category": ["레버리지"]},
            {"query": "나스닥100 vs 코스닥150 변동성 비교", "expected_category": ["해외주식", "국내주식"]},
        ]
    },
}

with open("project2_data/query_set.json", "w") as f:
    json.dump(personas, f, ensure_ascii=False, indent=2)

print(f"✅ {sum(len(p['queries']) for p in personas.values())}개 질의 저장 완료")

---
## 문제 2: 추가 ETF 문서 생성

In [ ]:
# ✅ 문제 2 정답
additional_etfs = [
    {"name": "TIGER ESG리더스", "category": "ESG", "market": "국내"},
    {"name": "KODEX 2차전지산업", "category": "2차전지", "market": "국내"},
    {"name": "TIGER 헬스케어", "category": "헬스케어", "market": "국내"},
    {"name": "KODEX 한국부동산리츠인프라", "category": "리츠", "market": "국내"},
    {"name": "KODEX 골드선물", "category": "원자재", "market": "글로벌"},
]

risk_map = {
    "ESG": 3, "2차전지": 4, "헬스케어": 3, "리츠": 2, "원자재": 3,
    "인덱스": 2, "섹터": 4, "배당": 2, "레버리지": 5, "테마": 4, "채권": 1, "자산배분": 2
}

for etf in additional_etfs:
    prompt = f"""다음 ETF에 대한 상세 설명을 작성해주세요:
    - ETF명: {etf['name']}
    - 카테고리: {etf['category']}
    - 시장: {etf['market']}

    다음 항목을 포함해주세요:
    1. 투자 전략 (3-4문장)
    2. 주요 편입 종목 (5개)
    3. 수수료 및 비용 (총보수)
    4. 적합한 투자자 유형
    5. 주의사항

    한국어로 300-400자 내외로 작성해주세요."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    doc_text = response.choices[0].message.content
    etf_documents.append({
        "name": etf["name"],
        "category": etf["category"],
        "market": etf["market"],
        "content": doc_text
    })
    print(f"✅ {etf['name']} 문서 생성 완료 (위험도: {risk_map.get(etf['category'], '?')})")

df = pd.DataFrame(etf_documents)
print(f"\n카테고리별 수:\n{df['category'].value_counts()}")

---
## 문제 3: 멀티-관련성 질의 생성

In [ ]:
# LLM으로 평가용 질의-정답 쌍 생성 (예시코드)
import json
from openai import OpenAI

client = OpenAI()

with open("project2_data/raw/etf_documents.json") as f:
    etf_docs = json.load(f)

eval_dataset = []

# 각 ETF 문서에 대해 관련 질의 생성
for i, doc in enumerate(etf_docs[:10]):  # 처음 10개
    prompt = f"""다음 ETF 문서를 읽고, 이 ETF를 찾기 위한 자연스러운 질의 3개를 만들어주세요.

    ETF: {doc['name']}
    카테고리: {doc['category']}
    내용: {doc['content'][:300]}

    JSON 배열로 반환: ["질의1", "질의2", "질의3"]"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        response_format={"type": "json_object"}
    )

    try:
        result = json.loads(response.choices[0].message.content)
        queries = result.get("queries", result.get("질의", list(result.values())[0]))
        if isinstance(queries, list):
            for q in queries:
                eval_dataset.append({
                    "query": q,
                    "relevant_doc_ids": [i],
                    "relevant_doc_names": [doc["name"]],
                    "category": doc["category"]
                })
    except:
        pass
    print(f"✅ {doc['name']}: 질의 생성 완료")

with open("project2_data/evaluation/eval_queries.json", "w") as f:
    json.dump(eval_dataset, f, ensure_ascii=False, indent=2)

print(f"\n📊 총 {len(eval_dataset)}개 평가 질의 생성")
for item in eval_dataset[:3]:
    print(f"  Q: {item['query']}")
    print(f"  A: {item['relevant_doc_names']}\n")

In [ ]:
# ✅ 문제 3 정답
import matplotlib.pyplot as plt

multi_queries = [
    {
        "query": "분산투자에 좋은 안전한 포트폴리오 구성 추천",
        "relevant": [{"doc_id": 0, "score": 2}, {"doc_id": 8, "score": 3}, {"doc_id": 9, "score": 2}],
        "irrelevant": [6]
    },
    {
        "query": "미국 시장에 투자할 수 있는 ETF 비교",
        "relevant": [{"doc_id": 1, "score": 3}, {"doc_id": 2, "score": 3}, {"doc_id": 5, "score": 2}],
        "irrelevant": [3, 8]
    },
    {
        "query": "배당 수익과 안정성을 동시에 추구하는 ETF",
        "relevant": [{"doc_id": 4, "score": 3}, {"doc_id": 5, "score": 2}, {"doc_id": 8, "score": 2}],
        "irrelevant": [2, 6]
    },
]

# 카테고리 분포 시각화
cats = [item['category'] for item in eval_dataset]
cat_counts = pd.Series(cats).value_counts()

plt.figure(figsize=(8, 4))
cat_counts.plot(kind='bar')
plt.title('평가 질의 카테고리 분포')
plt.xlabel('카테고리')
plt.ylabel('질의 수')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# CSV 저장
pd.DataFrame(eval_dataset).to_csv("project2_data/evaluation/eval_queries.csv", index=False)
print("✅ CSV 저장 완료")

---
## 문제 4: MMR 검색 구현

In [ ]:
# ✅ 문제 4 정답
import time

def format_results(results, method_name):
    print(f"\n🔍 {method_name} 결과:")
    for i, item in enumerate(results, 1):
        if isinstance(item, tuple) and len(item) == 2:
            doc, score = item
            print(f"  {i}. [{score:.4f}] {doc.metadata['name']} ({doc.metadata['category']})")
        else:
            print(f"  {i}. {item.metadata['name']} ({item.metadata['category']})")

query = "분산 투자에 적합한 ETF"

# Similarity Search
start = time.time()
sim_results = vectorstore.similarity_search_with_score(query, k=5)
sim_time = time.time() - start
format_results(sim_results, f"Similarity Search ({sim_time:.3f}s)")

# MMR Search
start = time.time()
mmr_results = vectorstore.max_marginal_relevance_search(query, k=5, fetch_k=10)
mmr_time = time.time() - start
format_results([(doc, 0) for doc in mmr_results], f"MMR Search ({mmr_time:.3f}s)")

# k값별 카테고리 다양성 비교
print("\n📊 k값별 카테고리 다양성:")
for k in [3, 5, 10]:
    sim = vectorstore.similarity_search(query, k=k)
    mmr = vectorstore.max_marginal_relevance_search(query, k=k, fetch_k=max(k*2, 10))
    sim_cats = set(d.metadata['category'] for d in sim)
    mmr_cats = set(d.metadata['category'] for d in mmr)
    print(f"  k={k}: Similarity {len(sim_cats)}개 카테고리 vs MMR {len(mmr_cats)}개 카테고리")

---
## 문제 5: Precision@K와 Recall@K 구현

In [ ]:
# ✅ 문제 5 정답
import matplotlib.pyplot as plt

def precision_at_k(vectorstore, eval_data, k=5):
    precisions = []
    for item in eval_data:
        results = vectorstore.similarity_search(item["query"], k=k)
        retrieved_ids = [r.metadata["doc_id"] for r in results]
        relevant_retrieved = sum(1 for rid in retrieved_ids if rid in item["relevant_doc_ids"])
        precisions.append(relevant_retrieved / k)
    return np.mean(precisions)

def recall_at_k(vectorstore, eval_data, k=5):
    recalls = []
    for item in eval_data:
        results = vectorstore.similarity_search(item["query"], k=k)
        retrieved_ids = [r.metadata["doc_id"] for r in results]
        relevant_retrieved = sum(1 for rid in retrieved_ids if rid in item["relevant_doc_ids"])
        total_relevant = len(item["relevant_doc_ids"])
        recalls.append(relevant_retrieved / total_relevant if total_relevant > 0 else 0)
    return np.mean(recalls)

ks = range(1, 11)
hrs = [hit_rate_at_k(vectorstore, eval_data, k) for k in ks]
mrrs = [mrr_at_k(vectorstore, eval_data, k) for k in ks]
precs = [precision_at_k(vectorstore, eval_data, k) for k in ks]
recs = [recall_at_k(vectorstore, eval_data, k) for k in ks]

plt.figure(figsize=(10, 6))
plt.plot(ks, hrs, 'o-', label='Hit Rate')
plt.plot(ks, mrrs, 's-', label='MRR')
plt.plot(ks, precs, '^-', label='Precision')
plt.plot(ks, recs, 'D-', label='Recall')
plt.xlabel('K')
plt.ylabel('Score')
plt.title('검색 평가 지표 비교 (K=1~10)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 문제 6: NDCG 평가 리포트

In [ ]:
# ✅ 문제 6 정답
report = {'baseline': {}}

for k in [1, 3, 5, 10]:
    hr = hit_rate_at_k(vectorstore, eval_data, k)
    mrr = mrr_at_k(vectorstore, eval_data, k)

    ndcg_scores = []
    for item in eval_data:
        results = vs.similarity_search(item["query"], k=k)
        rels = [1 if r.metadata["doc_id"] in item["relevant_doc_ids"] else 0 for r in results]
        ndcg_scores.append(ndcg_at_k(rels, k))

    report['baseline'][f'k={k}'] = {
        'hit_rate': round(hr, 4),
        'mrr': round(mrr, 4),
        'ndcg': round(np.mean(ndcg_scores), 4)
    }

with open("project2_data/evaluation/baseline_report.json", "w") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"{'K':<6} | {'Hit Rate':>10} | {'MRR':>10} | {'NDCG':>10}")
print("-" * 45)
for k_str, metrics in report['baseline'].items():
    print(f"{k_str:<6} | {metrics['hit_rate']:>10.4f} | {metrics['mrr']:>10.4f} | {metrics['ndcg']:>10.4f}")

print(f"\n✅ baseline_report.json 저장 완료")

---
## 문제 7: BM25 vs FAISS 비교 평가

In [ ]:
# ✅ 문제 7 정답
def bm25_hit_rate(eval_data, k=5):
    hits = 0
    for item in eval_data:
        results = bm25_search(item["query"], k)
        retrieved_ids = [r["doc_id"] for r in results]
        if any(rid in item["relevant_doc_ids"] for rid in retrieved_ids):
            hits += 1
    return hits / len(eval_data)

def bm25_mrr(eval_data, k=5):
    rr_sum = 0
    for item in eval_data:
        results = bm25_search(item["query"], k)
        retrieved_ids = [r["doc_id"] for r in results]
        for rank, rid in enumerate(retrieved_ids, 1):
            if rid in item["relevant_doc_ids"]:
                rr_sum += 1.0 / rank
                break
    return rr_sum / len(eval_data)

print(f"{'방법':<12} | {'K':>3} | {'Hit Rate':>10} | {'MRR':>10}")
print("-" * 45)
for k in [1, 3, 5, 10]:
    faiss_hr = hit_rate_at_k(vectorstore, eval_data, k)
    faiss_mrr = mrr_at_k(vectorstore, eval_data, k)
    bm25_hr = bm25_hit_rate(eval_data, k)
    bm25_m = bm25_mrr(eval_data, k)
    print(f"{'FAISS':<12} | {k:>3} | {faiss_hr:>10.4f} | {faiss_mrr:>10.4f}")
    print(f"{'BM25':<12} | {k:>3} | {bm25_hr:>10.4f} | {bm25_m:>10.4f}")
    print("-" * 45)

---
## 문제 8: Alpha 최적화

In [ ]:
# ✅ 문제 8 정답
best_alpha, best_hr = 0, 0

print(f"{'Alpha':>6} | {'Hit Rate@5':>12}")
print("-" * 25)

for a in np.arange(0, 1.1, 0.1):
    a = round(a, 1)
    hits = 0
    for item in eval_data:
        results = hybrid_search(item['query'], alpha=a, k=5)
        retrieved_ids = [r[0] for r in results]
        if any(rid in item['relevant_doc_ids'] for rid in retrieved_ids):
            hits += 1
    hr = hits / len(eval_data)
    print(f"{a:>6.1f} | {hr:>12.4f}")
    if hr > best_hr:
        best_alpha, best_hr = a, hr

checkpoint = {'best_alpha': best_alpha, 'best_hr': round(best_hr, 4)}
with open("project2_data/checkpoints/hybrid_config.json", "w") as f:
    json.dump(checkpoint, f, ensure_ascii=False, indent=2)

print(f"\n🏆 최적 alpha: {best_alpha} (Hit Rate: {best_hr:.4f})")
print("✅ hybrid_config.json 저장 완료")

---
## 문제 9: 도메인 특화 동의어 사전

In [ ]:
# ✅ 문제 9 정답
finance_synonyms = {
    'ETF': ['상장지수펀드', '인덱스펀드', '지수추종'],
    '배당': ['분배금', '배당금', '인컴', '이자수익'],
    '안정': ['안전', '보수적', '저위험', '원금보존'],
    '성장': ['그로스', '공격적', '고수익', '고성장'],
    '미국': ['해외', '글로벌', '나스닥', 'S&P'],
}

def synonym_expand(query):
    expanded = [query]
    for keyword, synonyms in finance_synonyms.items():
        if keyword in query:
            for syn in synonyms:
                expanded.append(query.replace(keyword, syn))
    return expanded

# Baseline vs Synonym Hit Rate 비교
baseline_hits, synonym_hits = 0, 0
for item in eval_data:
    # Baseline
    results = hybrid_search(item['query'], alpha=0.5, k=5)
    if any(r[0] in item['relevant_doc_ids'] for r in results):
        baseline_hits += 1

    # Synonym expanded
    expanded = synonym_expand(item['query'])
    all_results = {}
    for eq in expanded:
        for doc_id, score, name in hybrid_search(eq, alpha=0.5, k=5):
            if doc_id not in all_results or score > all_results[doc_id][1]:
                all_results[doc_id] = (doc_id, score, name)
    top_results = sorted(all_results.values(), key=lambda x: x[1], reverse=True)[:5]
    if any(r[0] in item['relevant_doc_ids'] for r in top_results):
        synonym_hits += 1

print(f"Baseline Hit Rate: {baseline_hits/len(eval_data):.4f}")
print(f"Synonym  Hit Rate: {synonym_hits/len(eval_data):.4f}")

---
## 문제 10: 커스텀 Multi-Query Retriever

In [ ]:
# ✅ 문제 10 정답
from langchain.prompts import PromptTemplate

custom_prompt = PromptTemplate(
    input_variables=['question'],
    template="""당신은 ETF 금융 상품 검색 전문가입니다.
다음 질문을 서로 다른 관점에서 3가지로 재작성하세요.
각 질의는 ETF 검색에 최적화되어야 합니다.

원래 질문: {question}

재작성된 질의 (한 줄에 하나씩):"""
)

retriever_custom = MultiQueryRetriever.from_llm(
    retriever=vs.as_retriever(search_kwargs={"k": 5}),
    llm=llm,
    prompt=custom_prompt
)

results_custom = retriever_custom.invoke("노후 대비 안정적 투자")
print(f"🔍 커스텀 Multi-Query 결과: {len(results_custom)}개")
for doc in results_custom:
    print(f"  - {doc.metadata['name']}: {doc.page_content[:80]}...")

---
## 문제 11: 검색 비교 대시보드

In [ ]:
# ✅ 문제 11 정답
import gradio as gr

search_history = []

def full_comparison(query, top_k):
    top_k = int(top_k)
    search_history.append(query)

    output = f"🔍 질의: {query}\n{'='*60}\n\n"

    # FAISS
    faiss_results = vs.similarity_search_with_score(query, k=top_k)
    output += "📌 FAISS 벡터 검색:\n"
    for i, (doc, score) in enumerate(faiss_results, 1):
        output += f"  {i}. [{score:.4f}] {doc.metadata['name']} ({doc.metadata['category']})\n"

    # BM25
    bm25_results = bm25_search(query, top_k)
    output += "\n📌 BM25 키워드 검색:\n"
    for i, r in enumerate(bm25_results, 1):
        output += f"  {i}. [{r['score']:.4f}] {r['name']}\n"

    # Hybrid
    hybrid_results = hybrid_search(query, alpha=0.5, k=top_k)
    output += "\n📌 하이브리드 검색 (α=0.5):\n"
    for i, (doc_id, score, name) in enumerate(hybrid_results, 1):
        output += f"  {i}. [{score:.4f}] {name}\n"

    return output

def show_history():
    if not search_history:
        return "검색 이력이 없습니다."
    return "\n".join(f"{i+1}. {q}" for i, q in enumerate(search_history))

with gr.Blocks(title="ETF 검색 비교 대시보드") as demo:
    with gr.Tab("검색"):
        query_input = gr.Textbox(label="검색 질의", placeholder="예: 배당 수익률 높은 안전한 ETF")
        top_k_slider = gr.Slider(1, 10, value=5, step=1, label="결과 수 (K)")
        search_btn = gr.Button("검색")
        output = gr.Textbox(label="비교 결과", lines=20)
        search_btn.click(full_comparison, [query_input, top_k_slider], output)

    with gr.Tab("검색 이력"):
        history_btn = gr.Button("이력 조회")
        history_output = gr.Textbox(label="최근 검색 이력", lines=10)
        history_btn.click(show_history, [], history_output)

demo.launch(share=True)